# Practical 4 — Transformer from Scratch using PyTorch
**Name:** Paras Vishwakarma | **Roll No:** 48 | **Class:** C | **BE SEM-II 2025-26**

**Topics:** Multi-Head Self-Attention · Positional Encoding · Encoder · Decoder · Full Transformer

This notebook builds every component of the Transformer architecture from scratch using PyTorch, based on the original paper *'Attention Is All You Need'* (Vaswani et al., 2017).

## Step 1 — Install & Import Libraries

In [1]:
!pip install torch numpy

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import math

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {device}')

PyTorch version : 2.10.0+cpu
Device          : cpu


## Step 2 — Scaled Dot-Product Attention
The core operation: `Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V`

- **Q** (Query): What we are looking for
- **K** (Key): What each token offers
- **V** (Value): Actual content to retrieve

In [3]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention
    Args:
        Q: Query  (batch, heads, seq_len, d_k)
        K: Key    (batch, heads, seq_len, d_k)
        V: Value  (batch, heads, seq_len, d_v)
        mask: optional mask to prevent attending to certain positions
    Returns:
        output, attention_weights
    """
    d_k = Q.size(-1)  # dimension of key/query

    # Step 1: Compute dot product of Q and K^T, scale by sqrt(d_k)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    # Step 2: Apply mask (fill masked positions with -infinity so softmax gives ~0)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # Step 3: Softmax to get attention weights (each row sums to 1)
    attn_weights = F.softmax(scores, dim=-1)

    # Step 4: Weighted sum of Values
    output = torch.matmul(attn_weights, V)

    return output, attn_weights


# Quick test
batch, heads, seq_len, d_k = 2, 4, 5, 16
Q = torch.randn(batch, heads, seq_len, d_k)
K = torch.randn(batch, heads, seq_len, d_k)
V = torch.randn(batch, heads, seq_len, d_k)

out, weights = scaled_dot_product_attention(Q, K, V)
print(f'Attention output shape  : {out.shape}')
print(f'Attention weights shape : {weights.shape}')
print(f'Attention weights sum   : {weights.sum(-1)}  (each row sums to 1)')

Attention output shape  : torch.Size([2, 4, 5, 16])
Attention weights shape : torch.Size([2, 4, 5, 5])
Attention weights sum   : tensor([[[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]]])  (each row sums to 1)


## Step 3 — Multi-Head Attention
Runs attention multiple times in parallel with different learned projections, then concatenates and projects results.

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        """
        d_model   : total embedding dimension (e.g. 512)
        num_heads : number of parallel attention heads (e.g. 8)
        d_k = d_model / num_heads (e.g. 64 per head)
        """
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'

        self.d_model    = d_model
        self.num_heads  = num_heads
        self.d_k        = d_model // num_heads  # dimension per head

        # Linear projections for Q, K, V and final output
        self.W_q = nn.Linear(d_model, d_model, bias=False)  # Query projection
        self.W_k = nn.Linear(d_model, d_model, bias=False)  # Key projection
        self.W_v = nn.Linear(d_model, d_model, bias=False)  # Value projection
        self.W_o = nn.Linear(d_model, d_model)               # Output projection

    def split_heads(self, x):
        """Reshape (batch, seq, d_model) -> (batch, heads, seq, d_k)"""
        batch_size, seq_len, _ = x.size()
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        return x.transpose(1, 2)  # (batch, heads, seq, d_k)

    def forward(self, Q, K, V, mask=None):
        # 1. Linear projections
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        # 2. Scaled dot-product attention on each head
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)

        # 3. Concatenate heads: (batch, heads, seq, d_k) -> (batch, seq, d_model)
        batch_size, _, seq_len, _ = attn_output.size()
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, self.d_model)

        # 4. Final linear projection
        output = self.W_o(attn_output)
        return output, attn_weights


# Test
d_model, num_heads, seq_len, batch_size = 512, 8, 10, 2
mha = MultiHeadAttention(d_model, num_heads)
x   = torch.randn(batch_size, seq_len, d_model)
out, w = mha(x, x, x)  # self-attention: Q=K=V
print(f'MultiHead Attention output shape  : {out.shape}')
print(f'Attention weights shape           : {w.shape}')
print(f'Parameters in MHA                 : {sum(p.numel() for p in mha.parameters()):,}')

MultiHead Attention output shape  : torch.Size([2, 10, 512])
Attention weights shape           : torch.Size([2, 8, 10, 10])
Parameters in MHA                 : 1,049,088


## Step 4 — Positional Encoding
Since Transformers have no recurrence, we add positional information using sine and cosine functions:

`PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))`  
`PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))`

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=5000, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Build the positional encoding matrix
        pe = torch.zeros(max_seq_len, d_model)          # (max_len, d_model)
        position = torch.arange(0, max_seq_len).unsqueeze(1).float()  # (max_len, 1)

        # Frequency divisor: 10000^(2i/d_model)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)   # even indices -> sin
        pe[:, 1::2] = torch.cos(position * div_term)   # odd indices  -> cos

        pe = pe.unsqueeze(0)  # (1, max_len, d_model) for broadcasting
        self.register_buffer('pe', pe)  # not a trainable parameter

    def forward(self, x):
        # Add positional encoding to input embeddings
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# Test
pos_enc = PositionalEncoding(d_model=512, max_seq_len=100)
x       = torch.randn(2, 20, 512)  # (batch=2, seq_len=20, d_model=512)
out     = pos_enc(x)
print(f'Positional Encoding input  shape : {x.shape}')
print(f'Positional Encoding output shape : {out.shape}')
print(f'PE values (pos 0, first 8 dims)  : {pos_enc.pe[0, 0, :8].tolist()}')
print(f'PE values (pos 1, first 8 dims)  : {pos_enc.pe[0, 1, :8].tolist()}')

Positional Encoding input  shape : torch.Size([2, 20, 512])
Positional Encoding output shape : torch.Size([2, 20, 512])
PE values (pos 0, first 8 dims)  : [0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0]
PE values (pos 1, first 8 dims)  : [0.8414709568023682, 0.5403023362159729, 0.8218562006950378, 0.5696950554847717, 0.8019617795944214, 0.5973753333091736, 0.7818871140480042, 0.6234200596809387]


## Step 5 — Position-wise Feed Forward Network
Applied to each position independently: `FFN(x) = max(0, xW1 + b1)W2 + b2`

In [6]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        d_model : input/output dimension (e.g. 512)
        d_ff    : inner/hidden dimension (typically 4 * d_model = 2048)
        """
        super(PositionWiseFeedForward, self).__init__()
        self.fc1     = nn.Linear(d_model, d_ff)
        self.fc2     = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        x = self.dropout(F.relu(self.fc1(x)))  # expand: d_model -> d_ff
        x = self.fc2(x)                         # compress: d_ff -> d_model
        return x


# Test
ffn = PositionWiseFeedForward(d_model=512, d_ff=2048)
x   = torch.randn(2, 10, 512)
out = ffn(x)
print(f'FFN input  shape : {x.shape}')
print(f'FFN output shape : {out.shape}')
print(f'FFN parameters   : {sum(p.numel() for p in ffn.parameters()):,}')

FFN input  shape : torch.Size([2, 10, 512])
FFN output shape : torch.Size([2, 10, 512])
FFN parameters   : 2,099,712


## Step 6 — Encoder Layer
Each Encoder Layer = **Multi-Head Self-Attention** + **Feed Forward** + **Layer Norm** + **Residual Connections**

In [7]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(EncoderLayer, self).__init__()

        self.self_attn  = MultiHeadAttention(d_model, num_heads)
        self.ffn        = PositionWiseFeedForward(d_model, d_ff, dropout)

        self.norm1      = nn.LayerNorm(d_model)  # after self-attention
        self.norm2      = nn.LayerNorm(d_model)  # after FFN

        self.dropout    = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        # 1. Self-Attention with residual connection + layer norm
        attn_out, _ = self.self_attn(x, x, x, src_mask)
        x = self.norm1(x + self.dropout(attn_out))   # Add & Norm

        # 2. Feed-Forward with residual connection + layer norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))    # Add & Norm

        return x


# Test
enc_layer = EncoderLayer(d_model=512, num_heads=8, d_ff=2048)
x         = torch.randn(2, 10, 512)
out       = enc_layer(x)
print(f'Encoder Layer input  shape : {x.shape}')
print(f'Encoder Layer output shape : {out.shape}')
print(f'Encoder Layer parameters   : {sum(p.numel() for p in enc_layer.parameters()):,}')

Encoder Layer input  shape : torch.Size([2, 10, 512])
Encoder Layer output shape : torch.Size([2, 10, 512])
Encoder Layer parameters   : 3,150,848


## Step 7 — Decoder Layer
Each Decoder Layer = **Masked Self-Attention** + **Cross-Attention** (encoder output) + **Feed Forward**

In [8]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(DecoderLayer, self).__init__()

        self.self_attn   = MultiHeadAttention(d_model, num_heads)  # masked self-attn
        self.cross_attn  = MultiHeadAttention(d_model, num_heads)  # encoder-decoder attn
        self.ffn         = PositionWiseFeedForward(d_model, d_ff, dropout)

        self.norm1       = nn.LayerNorm(d_model)
        self.norm2       = nn.LayerNorm(d_model)
        self.norm3       = nn.LayerNorm(d_model)

        self.dropout     = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        # 1. Masked Self-Attention (decoder can't see future tokens)
        self_attn_out, _ = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attn_out))

        # 2. Cross-Attention (Q from decoder, K & V from encoder)
        cross_attn_out, _ = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(cross_attn_out))

        # 3. Feed-Forward
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn_out))

        return x


# Test
dec_layer  = DecoderLayer(d_model=512, num_heads=8, d_ff=2048)
tgt        = torch.randn(2, 8, 512)   # decoder input (target sequence)
enc_out    = torch.randn(2, 10, 512)  # encoder output (source)
out        = dec_layer(tgt, enc_out)
print(f'Decoder Layer input  shape : {tgt.shape}')
print(f'Decoder Layer output shape : {out.shape}')
print(f'Decoder Layer parameters   : {sum(p.numel() for p in dec_layer.parameters()):,}')

Decoder Layer input  shape : torch.Size([2, 8, 512])
Decoder Layer output shape : torch.Size([2, 8, 512])
Decoder Layer parameters   : 4,200,960


## Step 8 — Full Transformer Model
Stacking N Encoder layers + N Decoder layers with embeddings and positional encoding.

In [9]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size,
                 d_model=512, num_heads=8, num_layers=6,
                 d_ff=2048, max_seq_len=5000, dropout=0.1):
        super(Transformer, self).__init__()

        # Embedding layers
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)

        # Positional encoding
        self.pos_encoding  = PositionalEncoding(d_model, max_seq_len, dropout)

        # Encoder: N stacked encoder layers
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # Decoder: N stacked decoder layers
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # Final output projection to vocabulary
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

        self.d_model  = d_model
        self.dropout  = nn.Dropout(dropout)

    def make_src_mask(self, src, pad_idx=0):
        """Mask padding tokens in source sequence."""
        # (batch, 1, 1, src_len)
        src_mask = (src != pad_idx).unsqueeze(1).unsqueeze(2)
        return src_mask

    def make_tgt_mask(self, tgt, pad_idx=0):
        """Causal mask: prevent decoder from looking at future positions."""
        tgt_len  = tgt.size(1)
        # Padding mask
        pad_mask = (tgt != pad_idx).unsqueeze(1).unsqueeze(2)  # (batch,1,1,tgt_len)
        # Causal (triangular) mask
        causal_mask = torch.tril(torch.ones(tgt_len, tgt_len, device=tgt.device)).bool()
        tgt_mask = pad_mask & causal_mask  # combine both
        return tgt_mask

    def encode(self, src, src_mask):
        # Embed + positional encode source
        x = self.pos_encoding(self.src_embedding(src) * math.sqrt(self.d_model))
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        return x

    def decode(self, tgt, enc_output, src_mask, tgt_mask):
        # Embed + positional encode target
        x = self.pos_encoding(self.tgt_embedding(tgt) * math.sqrt(self.d_model))
        for layer in self.decoder_layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        return x

    def forward(self, src, tgt):
        src_mask   = self.make_src_mask(src)
        tgt_mask   = self.make_tgt_mask(tgt)
        enc_output = self.encode(src, src_mask)
        dec_output = self.decode(tgt, enc_output, src_mask, tgt_mask)
        output     = self.fc_out(dec_output)  # (batch, tgt_len, vocab_size)
        return output


# Build model with small config for testing
model = Transformer(
    src_vocab_size = 1000,
    tgt_vocab_size = 1000,
    d_model        = 256,
    num_heads      = 8,
    num_layers     = 3,
    d_ff           = 512,
    max_seq_len    = 100,
    dropout        = 0.1
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable:,}')
print(f'Model structure:\n{model}')

Total parameters     : 4,715,752
Trainable parameters : 4,715,752
Model structure:
Transformer(
  (src_embedding): Embedding(1000, 256)
  (tgt_embedding): Embedding(1000, 256)
  (pos_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder_layers): ModuleList(
    (0-2): 3 x EncoderLayer(
      (self_attn): MultiHeadAttention(
        (W_q): Linear(in_features=256, out_features=256, bias=False)
        (W_k): Linear(in_features=256, out_features=256, bias=False)
        (W_v): Linear(in_features=256, out_features=256, bias=False)
        (W_o): Linear(in_features=256, out_features=256, bias=True)
      )
      (ffn): PositionWiseFeedForward(
        (fc1): Linear(in_features=256, out_features=512, bias=True)
        (fc2): Linear(in_features=512, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((256,), eps=1e-05, elementwis

## Step 9 — Forward Pass Test

In [10]:
# Simulate a batch of token sequences
batch_size = 4
src_len    = 12   # source sequence length
tgt_len    = 10   # target sequence length
vocab_size = 1000

# Random token indices (simulating tokenized sentences)
src = torch.randint(1, vocab_size, (batch_size, src_len)).to(device)  # avoid 0 (pad)
tgt = torch.randint(1, vocab_size, (batch_size, tgt_len)).to(device)

# Forward pass
model.eval()
with torch.no_grad():
    output = model(src, tgt)

print(f'Source shape         : {src.shape}   (batch, src_len)')
print(f'Target shape         : {tgt.shape}   (batch, tgt_len)')
print(f'Output shape         : {output.shape}  (batch, tgt_len, vocab_size)')
print(f'\nOutput[0][0] (logits for first token of first sentence):')
print(output[0][0][:10])  # first 10 logits

Source shape         : torch.Size([4, 12])   (batch, src_len)
Target shape         : torch.Size([4, 10])   (batch, tgt_len)
Output shape         : torch.Size([4, 10, 1000])  (batch, tgt_len, vocab_size)

Output[0][0] (logits for first token of first sentence):
tensor([ 0.7027, -0.0206,  0.1356,  0.4646,  0.9335, -0.1128,  0.0880, -0.1960,
         0.2934,  0.6808])


## Step 10 — Training Loop on a Simple Task
Training the Transformer on a **copy task**: the model learns to output the same sequence as input (simple sanity check).

In [11]:
# Hyperparameters
VOCAB_SIZE  = 50
D_MODEL     = 64
NUM_HEADS   = 4
NUM_LAYERS  = 2
D_FF        = 128
SEQ_LEN     = 8
BATCH_SIZE  = 32
EPOCHS      = 30
LR          = 0.001

# Build small model for copy task
copy_model = Transformer(
    src_vocab_size = VOCAB_SIZE,
    tgt_vocab_size = VOCAB_SIZE,
    d_model        = D_MODEL,
    num_heads      = NUM_HEADS,
    num_layers     = NUM_LAYERS,
    d_ff           = D_FF,
    max_seq_len    = 100,
    dropout        = 0.1
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore padding
optimizer = optim.Adam(copy_model.parameters(), lr=LR, betas=(0.9, 0.98), eps=1e-9)

print(f'Model parameters : {sum(p.numel() for p in copy_model.parameters()):,}')
print(f'Training for {EPOCHS} epochs...')

def generate_batch(batch_size, seq_len, vocab_size):
    """Generate a batch for copy task: src and tgt are the same random sequences."""
    src = torch.randint(1, vocab_size, (batch_size, seq_len))
    tgt = src.clone()
    return src.to(device), tgt.to(device)

losses = []

copy_model.train()
for epoch in range(EPOCHS):
    src, tgt = generate_batch(BATCH_SIZE, SEQ_LEN, VOCAB_SIZE)

    # Teacher forcing: feed tgt[:-1] as input, predict tgt[1:]
    tgt_input  = tgt[:, :-1]  # (batch, seq-1)
    tgt_output = tgt[:, 1:]   # (batch, seq-1)

    optimizer.zero_grad()
    output = copy_model(src, tgt_input)  # (batch, seq-1, vocab)

    # Reshape for loss: (batch * seq, vocab) vs (batch * seq)
    loss = criterion(
        output.reshape(-1, VOCAB_SIZE),
        tgt_output.reshape(-1)
    )
    loss.backward()

    # Gradient clipping (prevents exploding gradients)
    torch.nn.utils.clip_grad_norm_(copy_model.parameters(), max_norm=1.0)

    optimizer.step()
    losses.append(loss.item())

    if (epoch + 1) % 5 == 0:
        print(f'  Epoch [{epoch+1:3d}/{EPOCHS}]  Loss: {loss.item():.4f}')

print(f'\nTraining complete! Final Loss: {losses[-1]:.4f}')

Model parameters : 175,922
Training for 30 epochs...
  Epoch [  5/30]  Loss: 4.0128
  Epoch [ 10/30]  Loss: 3.9189
  Epoch [ 15/30]  Loss: 3.9174
  Epoch [ 20/30]  Loss: 3.8643
  Epoch [ 25/30]  Loss: 3.7768
  Epoch [ 30/30]  Loss: 3.7951

Training complete! Final Loss: 3.7951


In [12]:
# Loss curve
print('Training Loss per Epoch:')
for i, l in enumerate(losses):
    bar = '#' * int(l * 10)
    print(f'  Epoch {i+1:2d}: {l:.4f}  {bar}')

Training Loss per Epoch:
  Epoch  1: 4.0506  ########################################
  Epoch  2: 3.9738  #######################################
  Epoch  3: 4.0051  ########################################
  Epoch  4: 4.0514  ########################################
  Epoch  5: 4.0128  ########################################
  Epoch  6: 4.0430  ########################################
  Epoch  7: 3.9850  #######################################
  Epoch  8: 3.9609  #######################################
  Epoch  9: 3.9884  #######################################
  Epoch 10: 3.9189  #######################################
  Epoch 11: 3.9270  #######################################
  Epoch 12: 3.9233  #######################################
  Epoch 13: 3.9567  #######################################
  Epoch 14: 3.8914  ######################################
  Epoch 15: 3.9174  #######################################
  Epoch 16: 3.8794  ######################################
  Epoch 17: 

## Step 11 — Model Architecture Summary

In [13]:
# Architecture summary table
import pandas as pd

arch_df = pd.DataFrame({
    'Component'   : [
        'Source Embedding', 'Target Embedding', 'Positional Encoding',
        'Encoder Layer x N', '  └ Multi-Head Self-Attention', '  └ Feed Forward Network',
        '  └ Layer Norm x2 + Dropout',
        'Decoder Layer x N', '  └ Masked Multi-Head Self-Attention',
        '  └ Cross-Attention (Enc-Dec)', '  └ Feed Forward Network',
        '  └ Layer Norm x3 + Dropout',
        'Output Linear Projection'
    ],
    'Input -> Output Shape' : [
        '(batch, src_len) -> (batch, src_len, d_model)',
        '(batch, tgt_len) -> (batch, tgt_len, d_model)',
        '(batch, seq, d_model) -> (batch, seq, d_model)',
        'Repeated N times', 'Q=K=V=x -> (batch, seq, d_model)',
        '(batch, seq, d_model) -> (batch, seq, d_model)',
        'Residual + Normalize',
        'Repeated N times',
        'Q=K=V=tgt -> (batch, tgt, d_model)',
        'Q=tgt, K=V=enc -> (batch, tgt, d_model)',
        '(batch, tgt, d_model) -> (batch, tgt, d_model)',
        'Residual + Normalize',
        '(batch, tgt_len, d_model) -> (batch, tgt_len, vocab_size)'
    ]
})
print('=== Transformer Architecture Summary ===')
arch_df

=== Transformer Architecture Summary ===


,Component,Input -> Output Shape
0,Source Embedding,"(batch, src_len) -> (batch, src_len, d_model)"
1,Target Embedding,"(batch, tgt_len) -> (batch, tgt_len, d_model)"
2,Positional Encoding,"(batch, seq, d_model) -> (batch, seq, d_model)"
3,Encoder Layer x N,Repeated N times
4,└ Multi-Head Self-Attention,"Q=K=V=x -> (batch, seq, d_model)"
5,└ Feed Forward Network,"(batch, seq, d_model) -> (batch, seq, d_model)"
6,└ Layer Norm x2 + Dropout,Residual + Normalize
7,Decoder Layer x N,Repeated N times
8,└ Masked Multi-Head Self-Attention,"Q=K=V=tgt -> (batch, tgt, d_model)"
9,└ Cross-Attention (Enc-Dec),"Q=tgt, K=V=enc -> (batch, tgt, d_model)"


In [14]:
# Save the model
torch.save(copy_model.state_dict(), 'transformer_model.pth')
print('Model saved as transformer_model.pth')

# Verify by loading back
loaded_model = Transformer(
    src_vocab_size=VOCAB_SIZE, tgt_vocab_size=VOCAB_SIZE,
    d_model=D_MODEL, num_heads=NUM_HEADS, num_layers=NUM_LAYERS,
    d_ff=D_FF, max_seq_len=100, dropout=0.1
).to(device)
loaded_model.load_state_dict(torch.load('transformer_model.pth', map_location=device))
loaded_model.eval()
print('Model loaded successfully!')

# Quick inference test
test_src, test_tgt = generate_batch(1, SEQ_LEN, VOCAB_SIZE)
with torch.no_grad():
    test_out = loaded_model(test_src, test_tgt[:, :-1])
predicted = test_out.argmax(-1)
print(f'\nTest Source    : {test_src[0].tolist()}')
print(f'Test Target    : {test_tgt[0, 1:].tolist()}')
print(f'Model Predicted: {predicted[0].tolist()}')

Model saved as transformer_model.pth
Model loaded successfully!

Test Source    : [18, 27, 3, 21, 38, 8, 39, 29]
Test Target    : [27, 3, 21, 38, 8, 39, 29]
Model Predicted: [4, 7, 7, 18, 7, 7, 18]
